In [13]:
# ── Cell 1: Imports & paths ───────────────────────────────────────────────
import polars as pl
import pandas as pd
import numpy as np
from scipy.optimize import brentq
from statsmodels.nonparametric.smoothers_lowess import lowess
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

Y_MEDIAN      = 10_784
YIELD_FLOOR   = 7_000    # operational minimum
YIELD_CEILING = 15_000   # DNQ above this

In [14]:
df_supply = pl.read_parquet(PROCESSED_DIR / "jan2026_od_supply.parquet")
df_quotes = pl.read_parquet(PROCESSED_DIR / "jan2026_may2026_quotes_forecast.parquet")

with open(PROCESSED_DIR / "elasticity_params.json") as f:
    params = json.load(f)

print("Supply schema:"); print(df_supply.schema)
print("\nQuotes schema:"); print(df_quotes.schema)
print(f"\nSupply shape: {df_supply.shape}, Quotes shape: {df_quotes.shape}")
print(f"\nElasticity params loaded:")
print(f"  Y_median = {params['Y_median']}")
print(f"  a = {params['conversion_elasticity']['a']:.4f}")
print(f"  b = {params['conversion_elasticity']['b']:.4f}")
print(f"  LOWESS points = {len(params['quote_count_elasticity']['x'])}")

Supply schema:
Schema({'date': Datetime(time_unit='us', time_zone=None), 'num_tails': Int64, 'delivered_tails_fc': Float64, 'raw_leg_supply_fc': Float64, 'program_live_legs_fc': Float64, 'non_program_live_legs_fc': Float64, 'current_od_adjusted': Float64, 'od_supply': Float64})

Quotes schema:
Schema({'date': Date, 'forecasted_quotes': Float64, 'yhat_lower': Float64, 'yhat_upper': Float64})

Supply shape: (151, 8), Quotes shape: (151, 4)

Elasticity params loaded:
  Y_median = 10784
  a = 0.2297
  b = 0.1736
  LOWESS points = 16


In [15]:
# ── Cell 3 Reconstruct elasticity functions: PATCHED: D(Y) in booking units ────────────────────────────────
# ε_q — LOWESS interpolation
x_low      = np.array(params["quote_count_elasticity"]["x"])
y_low_norm = np.array(params["quote_count_elasticity"]["y_normalized"])

def eps_q(Y):
    # Clamp to 1.0 outside LOWESS range (inelastic assumption at tails)
    if Y < x_low.min(): return float(np.interp(x_low.min(), x_low, y_low_norm))
    if Y > x_low.max(): return float(np.interp(x_low.max(), x_low, y_low_norm))
    return float(np.interp(Y, x_low, y_low_norm))

# ε_c — exponential decay, Y rescaled to $k
a_fit          = params["conversion_elasticity"]["a"]
b_fit          = params["conversion_elasticity"]["b"]
conv_at_median = a_fit * np.exp(-b_fit * Y_MEDIAN / 1_000)

def eps_c(Y):
    return (a_fit * np.exp(-b_fit * Y / 1_000)) / conv_at_median

# FIXED: D returns expected bookings (comparable to od_supply in legs)
def D(Y, forecasted_quotes):
    raw_conv = a_fit * np.exp(-b_fit * Y / 1_000)   # actual conversion rate at Y
    return forecasted_quotes * eps_q(Y) * raw_conv

# Sanity checks
print("D(Y) sanity checks (should be in booking units now):")
print(f"  D(Y_median, 80.3)  = {D(Y_MEDIAN, 80.3):.2f}  (expect ~2.8 bookings)")
print(f"  D(floor,    80.3)  = {D(YIELD_FLOOR, 80.3):.2f}  (expect ~5-8 bookings)")
print(f"  D(ceiling,  80.3)  = {D(YIELD_CEILING, 80.3):.2f}  (expect ~0.5-1.5 bookings)")
print(f"  Mean od_supply     = 1.50  (solver should find crossing in floor-ceiling range)")

D(Y) sanity checks (should be in booking units now):
  D(Y_median, 80.3)  = 2.84  (expect ~2.8 bookings)
  D(floor,    80.3)  = 5.12  (expect ~5-8 bookings)
  D(ceiling,  80.3)  = 0.66  (expect ~0.5-1.5 bookings)
  Mean od_supply     = 1.50  (solver should find crossing in floor-ceiling range)


In [16]:
# ── Cell 4: Merge supply + quotes (PATCHED — cast datetime to date) ────────
df_solver = (
    df_supply
    .with_columns(pl.col("date").cast(pl.Date))   # ← Datetime → Date to match quotes
    .join(
        df_quotes.select(["date", "forecasted_quotes"]),
        on="date", how="left"
    )
    .sort("date")
)

print(df_solver.select(["date", "od_supply", "forecasted_quotes"]))
print(f"\nZero supply days : {(df_solver['od_supply'] == 0).sum()}")
print(f"Non-zero supply days : {(df_solver['od_supply'] > 0).sum()}")
print(f"Null forecasted_quotes (join miss): {df_solver['forecasted_quotes'].is_null().sum()}")

shape: (151, 3)
┌────────────┬───────────┬───────────────────┐
│ date       ┆ od_supply ┆ forecasted_quotes │
│ ---        ┆ ---       ┆ ---               │
│ date       ┆ f64       ┆ f64               │
╞════════════╪═══════════╪═══════════════════╡
│ 2026-01-01 ┆ 0.0       ┆ 77.41             │
│ 2026-01-02 ┆ 0.0       ┆ 71.19             │
│ 2026-01-03 ┆ 5.702237  ┆ 37.26             │
│ 2026-01-04 ┆ 0.0       ┆ 36.97             │
│ 2026-01-05 ┆ 1.22002   ┆ 94.17             │
│ …          ┆ …         ┆ …                 │
│ 2026-05-27 ┆ 0.0       ┆ 64.69             │
│ 2026-05-28 ┆ 0.0       ┆ 56.3              │
│ 2026-05-29 ┆ 0.0       ┆ 48.33             │
│ 2026-05-30 ┆ 4.050872  ┆ 14.01             │
│ 2026-05-31 ┆ 0.0       ┆ 11.76             │
└────────────┴───────────┴───────────────────┘

Zero supply days : 120
Non-zero supply days : 31
Null forecasted_quotes (join miss): 0


In [17]:
# ── Cell 5: Solver loop ───────────────────────────────────────────────────
results = []

for row in df_solver.iter_rows(named=True):
    date             = row["date"]
    supply           = row["od_supply"]
    fcast_quotes     = row["forecasted_quotes"]

    if supply == 0:
        # Zero supply → DNQ signal → ceiling yield
        optimal_yield  = YIELD_CEILING
        d_at_yield     = 0.0
        solver_status  = "DNQ_zero_supply"

    else:
        # Check if demand at floor already <= supply (extremely tight market)
        d_at_floor   = D(YIELD_FLOOR, fcast_quotes)
        d_at_ceiling = D(YIELD_CEILING, fcast_quotes)

        if d_at_floor <= supply:
            # Even at floor price demand doesn't exceed supply → price at floor
            optimal_yield = YIELD_FLOOR
            d_at_yield    = d_at_floor
            solver_status = "demand_below_floor"

        elif d_at_ceiling >= supply:
            # Demand still exceeds supply even at ceiling → price at ceiling
            optimal_yield = YIELD_CEILING
            d_at_yield    = d_at_ceiling
            solver_status = "demand_above_ceiling"

        else:
            # Normal case: brentq finds where D(Y) - supply = 0
            try:
                optimal_yield = brentq(
                    lambda Y: D(Y, fcast_quotes) - supply,
                    YIELD_FLOOR, YIELD_CEILING,
                    xtol=1.0,      # $1 precision is sufficient
                    maxiter=200
                )
                d_at_yield    = D(optimal_yield, fcast_quotes)
                solver_status = "solved"
            except ValueError as e:
                optimal_yield = Y_MEDIAN
                d_at_yield    = D(Y_MEDIAN, fcast_quotes)
                solver_status = f"error: {e}"

    results.append({
        "date"            : date,
        "od_supply"       : supply,
        "forecasted_quotes": fcast_quotes,
        "optimal_yield"   : round(optimal_yield, 0),
        "d_at_yield"      : round(d_at_yield, 2),
        "solver_status"   : solver_status
    })

df_results = pl.DataFrame(results)
print(df_results)
print(f"\nSolver status breakdown:")
print(df_results.group_by("solver_status").agg(pl.len().alias("days")).sort("days", descending=True))

shape: (151, 6)
┌────────────┬───────────┬───────────────────┬───────────────┬────────────┬────────────────────┐
│ date       ┆ od_supply ┆ forecasted_quotes ┆ optimal_yield ┆ d_at_yield ┆ solver_status      │
│ ---        ┆ ---       ┆ ---               ┆ ---           ┆ ---        ┆ ---                │
│ date       ┆ f64       ┆ f64               ┆ f64           ┆ f64        ┆ str                │
╞════════════╪═══════════╪═══════════════════╪═══════════════╪════════════╪════════════════════╡
│ 2026-01-01 ┆ 0.0       ┆ 77.41             ┆ 15000.0       ┆ 0.0        ┆ DNQ_zero_supply    │
│ 2026-01-02 ┆ 0.0       ┆ 71.19             ┆ 15000.0       ┆ 0.0        ┆ DNQ_zero_supply    │
│ 2026-01-03 ┆ 5.702237  ┆ 37.26             ┆ 7000.0        ┆ 2.38       ┆ demand_below_floor │
│ 2026-01-04 ┆ 0.0       ┆ 36.97             ┆ 15000.0       ┆ 0.0        ┆ DNQ_zero_supply    │
│ 2026-01-05 ┆ 1.22002   ┆ 94.17             ┆ 13798.0       ┆ 1.22       ┆ solved             │
│ …          ┆

In [18]:
df_plot = df_results.to_pandas()
df_plot["date"] = pd.to_datetime(df_plot["date"])

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        "Jan-May 2026 — Optimal Yield Recommendation by Date",
        "Jan-May 2026 — OD Supply vs Forecasted Quotes"
    ],
    vertical_spacing=0.12
)

# ── Colour-code by solver status ──────────────────────────────────────────
color_map = {
    "solved"              : "steelblue",
    "DNQ_zero_supply"     : "crimson",
    "demand_above_ceiling": "darkorange",
    "demand_below_floor"  : "green"
}

for status, colour in color_map.items():
    mask = df_plot["solver_status"] == status
    if mask.sum() == 0:
        continue
    fig.add_trace(go.Bar(
        x=df_plot.loc[mask, "date"],
        y=df_plot.loc[mask, "optimal_yield"],
        name=status,
        marker_color=colour,
        opacity=0.85
    ), row=1, col=1)

# Reference lines on yield panel
fig.add_hline(y=Y_MEDIAN,      line_dash="dash", line_color="green",
              annotation_text=f"Y_median=${Y_MEDIAN:,}", row=1, col=1)
fig.add_hline(y=YIELD_CEILING, line_dash="dot",  line_color="red",
              annotation_text=f"DNQ ceiling=${YIELD_CEILING:,}", row=1, col=1)
fig.add_hline(y=YIELD_FLOOR,   line_dash="dot",  line_color="grey",
              annotation_text=f"Floor=${YIELD_FLOOR:,}", row=1, col=1)

# ── Supply vs quotes panel ────────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=df_plot["date"], y=df_plot["od_supply"],
    name="OD Supply (legs)", marker_color="steelblue", opacity=0.7
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df_plot["date"], y=df_plot["forecasted_quotes"],
    mode="lines+markers", name="Forecasted quotes",
    line=dict(color="darkorange", width=2), marker=dict(size=5)
), row=2, col=1)

fig.update_layout(
    width=1200, height=700,
    title="09 — Dynamic Pricing Engine: Jan-May 2026 Yield Recommendations",
    barmode="stack",
    legend=dict(orientation="h", x=0.5, xanchor="center",
                y=-0.08, font=dict(size=10)),
    margin=dict(l=50, r=50, t=50, b=50)
)
fig.update_yaxes(title_text="Optimal Yield ($/EFT hr)", tickformat="$,.0f", row=1, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()
fig.write_html("./09_solver.html")
print("Saved → ./09_solver.html")

Saved → ./09_solver.html


In [19]:
df_results.write_parquet(PROCESSED_DIR / "optimal_yield_per_date.parquet")
print("✅ Saved → data/processed/optimal_yield_per_date.parquet")

# Summary stats
solved = df_results.filter(pl.col("solver_status") == "solved")
print(f"\n── Jan-May 2026 Yield Recommendations ──")
print(f"  DNQ days (zero supply)    : {(df_results['solver_status'] == 'DNQ_zero_supply').sum()}")
print(f"  Solved days               : {(df_results['solver_status'] == 'solved').sum()}")
print(f"  Ceiling days              : {(df_results['solver_status'] == 'demand_above_ceiling').sum()}")
print(f"  Floor days                : {(df_results['solver_status'] == 'demand_below_floor').sum()}")
if len(solved) > 0:
    print(f"\n  Solved yield — mean : ${solved['optimal_yield'].mean():,.0f}")
    print(f"  Solved yield — min  : ${solved['optimal_yield'].min():,.0f}")
    print(f"  Solved yield — max  : ${solved['optimal_yield'].max():,.0f}")

print("\n── Per-Date Recommendations (first 10 and last 10) ──")
print(f"{'Date':<14} {'Supply':>8} {'Quotes':>8} {'Yield':>10} {'Status'}")
print("─" * 60)
sorted_results = df_results.sort("date")
for row in sorted_results.head(10).iter_rows(named=True):
    status_short = {
        "DNQ_zero_supply"   : "DNQ",
        "solved"            : "SOLVED",
        "demand_below_floor": "FLOOR",
        "demand_above_ceiling": "CEILING"
    }.get(row["solver_status"], row["solver_status"])
    print(f"{str(row['date']):<14} {row['od_supply']:>8.2f} "
          f"{row['forecasted_quotes']:>8.1f} "
          f"${row['optimal_yield']:>9,.0f}  {status_short}")
print("  ...")
for row in sorted_results.tail(10).iter_rows(named=True):
    status_short = {
        "DNQ_zero_supply"   : "DNQ",
        "solved"            : "SOLVED",
        "demand_below_floor": "FLOOR",
        "demand_above_ceiling": "CEILING"
    }.get(row["solver_status"], row["solver_status"])
    print(f"{str(row['date']):<14} {row['od_supply']:>8.2f} "
          f"{row['forecasted_quotes']:>8.1f} "
          f"${row['optimal_yield']:>9,.0f}  {status_short}")

✅ Saved → data/processed/optimal_yield_per_date.parquet

── Jan-May 2026 Yield Recommendations ──
  DNQ days (zero supply)    : 120
  Solved days               : 22
  Ceiling days              : 1
  Floor days                : 8

  Solved yield — mean : $13,139
  Solved yield — min  : $10,566
  Solved yield — max  : $14,936

── Per-Date Recommendations (first 10 and last 10) ──
Date             Supply   Quotes      Yield Status
────────────────────────────────────────────────────────────
2026-01-01         0.00     77.4 $   15,000  DNQ
2026-01-02         0.00     71.2 $   15,000  DNQ
2026-01-03         5.70     37.3 $    7,000  FLOOR
2026-01-04         0.00     37.0 $   15,000  DNQ
2026-01-05         1.22     94.2 $   13,798  SOLVED
2026-01-06         3.82    102.5 $   10,566  SOLVED
2026-01-07         2.01    100.3 $   12,593  SOLVED
2026-01-08         0.00     93.3 $   15,000  DNQ
2026-01-09         0.00     86.5 $   15,000  DNQ
2026-01-10         5.81     51.8 $    7,000  FLOOR
  ..